In [1]:
import json
import pandas as pd
from collections import Counter

# Load cleaned Tarla Dalal dataset
with open("final_cleaned_tarla_dalal.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Flatten all ingredients into one list
all_ingredients = []
for recipe in data:
    ingredients = recipe.get("ingredients", [])
    for item in ingredients:
        normalized = item.lower().strip()
        all_ingredients.append(normalized)

# Count frequency of ingredients
ingredient_counts = Counter(all_ingredients)

# Create DataFrame
df = pd.DataFrame(ingredient_counts.items(), columns=["ingredient_name", "count"])

# Add placeholder columns for flavor profile
for flavor in ["spicy", "sweet", "sour", "bitter", "salty"]:
    df[flavor] = 0  # Initially 0; you’ll manually fill top 100–150

# Sort by frequency
df = df.sort_values(by="count", ascending=False).reset_index(drop=True)

# Save to CSV for manual labeling
df.to_csv("top_ingredients_flavor_profile.csv", index=False, encoding="utf-8")

print("✅ Dataset created: top_ingredients_flavor_profile.csv")


✅ Dataset created: top_ingredients_flavor_profile.csv


In [2]:
import pandas as pd
import re

# Load your existing CSV
df = pd.read_csv("top_ingredients_flavor_profile.csv")

# Units and keywords to remove
units_to_remove = [
    "tsp", "tbsp", "cup", "cups", "ml", "ltr", "litre", "litres",
    "grams", "gram", "g", "kg", "pinch", "slice", "slices", "clove", "cloves",
    "piece", "pieces", "inch", "inches", "packet", "packets", "can", "cans"
]

# Cleaning function
def clean_ingredient(text):
    text = text.lower()
    text = re.sub(r"\b\d+([\/\.\d]*)?\b", "", text)  # remove integers, decimals, fractions
    text = re.sub(r"[\u00BC-\u00BE\u2150-\u215E]", "", text)  # remove unicode fractions
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # remove punctuation and symbols
    for unit in units_to_remove:
        text = re.sub(rf"\b{unit}\b", "", text)
    return re.sub(r"\s+", " ", text).strip()

# Apply cleaning
df["ingredient_name"] = df["ingredient_name"].apply(clean_ingredient)

# Group and sum duplicate ingredient names (after cleaning)
df = df.groupby("ingredient_name", as_index=False).agg({
    "count": "sum",
    "spicy": "max",
    "sweet": "max",
    "sour": "max",
    "bitter": "max",
    "salty": "max"
})

# Sort and save again
df = df.sort_values(by="count", ascending=False).reset_index(drop=True)
df.to_csv("top_ingredients_flavor_profile_cleaned.csv", index=False, encoding="utf-8")

print("✅ Cleaned and saved: top_ingredients_flavor_profile_cleaned.csv")


✅ Cleaned and saved: top_ingredients_flavor_profile_cleaned.csv
